In [58]:
import pandas as pd

In [60]:
hpi = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/hpi_master.csv"
)
hpi = hpi[
    (hpi["level"] == "MSA") & (hpi["yr"] >= 1999) & (hpi["yr"] <= 2020)
]  # only keep data for msa level
hpi = hpi[["place_name", "place_id", "yr", "period", "index_nsa"]]
hpi = hpi.groupby(["place_id", "yr"], as_index=False).agg(
    {
        "place_name": "first",
        "place_id": "first",
        "yr": "first",
        "index_nsa": "mean",
    }
)

hpi["index_prev"] = hpi.groupby(["place_id"])["index_nsa"].shift(1)
hpi["hpi_change"] = hpi["index_nsa"] - hpi["index_prev"]

hpi["yr"] = pd.to_numeric(hpi["yr"], errors="coerce")
hpi["index_nsa"] = pd.to_numeric(hpi["index_nsa"], errors="coerce")

hpi["hpi_yoy"] = (hpi["index_nsa"] / hpi["index_prev"] - 1) * 100

hpi = hpi.dropna(subset=["hpi_yoy"])

hpi.head()

,place_name,place_id,yr,index_nsa,index_prev,hpi_change,hpi_yoy
1,"Abilene, TX",10180,2000,115.7100,114.3075,1.4025,1.226954
2,"Abilene, TX",10180,2001,120.4200,115.7100,4.7100,4.070521
3,"Abilene, TX",10180,2002,123.8525,120.4200,3.4325,2.850440
4,"Abilene, TX",10180,2003,127.0975,123.8525,3.2450,2.620052
5,"Abilene, TX",10180,2004,132.7600,127.0975,5.6625,4.455241


In [61]:
xwalk = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/list1_2023.csv"
)
xwalk.columns = (
    xwalk.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
)
xwalk["county_fips5"] = (
    xwalk["fips_state_code"].astype(str).str.upper().str.strip().str.zfill(2)
    + xwalk["fips_county_code"].astype(str).str.zfill(3).str.strip()
)
xwalk = xwalk[
    [
        "cbsa_code",
        "cbsa_title",
        "county_fips5",
        "county/county_equivalent",
        "state_name",
    ]
]
xwalk["cbsa_code"] = xwalk["cbsa_code"].astype(str)
# xwalk.info()
xwalk

,cbsa_code,cbsa_title,county_fips5,county/county_equivalent,state_name
0,10100,"Aberdeen, SD",46013,Brown County,South Dakota
1,10100,"Aberdeen, SD",46045,Edmunds County,South Dakota
2,10140,"Aberdeen, WA",53027,Grays Harbor County,Washington
3,10180,"Abilene, TX",48059,Callahan County,Texas
4,10180,"Abilene, TX",48253,Jones County,Texas
...,...,...,...,...,...
1910,49700,"Yuba City, CA",06101,Sutter County,California
1911,49700,"Yuba City, CA",06115,Yuba County,California
1912,49740,"Yuma, AZ",04027,Yuma County,Arizona
1913,49780,"Zanesville, OH",39119,Muskingum County,Ohio


In [ ]:
hpi_county_level = xwalk.merge(
    hpi,
    left_on=["cbsa_code"],  # , "nri_ver"],
    right_on=["place_id"],  # , "yr"],
    how="left",
).dropna(subset="hpi_yoy")

hpi_county_level["yr"] = hpi_county_level["yr"].astype(int)
hpi_county_level

,cbsa_code,cbsa_title,county_fips5,county/county_equivalent,state_name,place_name,place_id,yr,index_nsa,index_prev,hpi_change,hpi_yoy
3,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2000,115.7100,114.3075,1.4025,1.226954
4,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2001,120.4200,115.7100,4.7100,4.070521
5,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2002,123.8525,120.4200,3.4325,2.850440
6,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2003,127.0975,123.8525,3.2450,2.620052
7,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2004,132.7600,127.0975,5.6625,4.455241
...,...,...,...,...,...,...,...,...,...,...,...,...
22827,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2016,171.2425,164.9825,6.2600,3.794342
22828,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2017,177.8425,171.2425,6.6000,3.854183
22829,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2018,184.3200,177.8425,6.4775,3.642268
22830,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2019,193.8850,184.3200,9.5650,5.189345


In [63]:
hpi_county_level.to_csv(
    "../../02_processed_data/hpi_county_2000.csv",
    index=False,
)